# **Analysis of Output with Radius Circle Mass**

This notebook is designed to analyze the output of cell tracking and intensity measurements using a radius circle mass approach.

### **Imports and Setup**
Import necessary libraries and initialize the Napari viewer.

In [ ]:
import os
import matplotlib.pyplot as plt
import napari
import tifffile
from tqdm import tqdm
from scipy import ndimage
import pickle
import numpy as np
seg_utils = __import__('0_cct_utils')

viewer = napari.Viewer()

### **Paths Setup**
Set up paths to data folders and define the base filename for processing.

In [ ]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
model_folder = "ModelAB1"
file_name = "OUA_060525_cluster1_20min_10i_340ms"

raw_path = os.path.join(parent_dir, "raw_data", model_folder)
mask_path = os.path.join(parent_dir, "masks_tracked", model_folder)
pkl_path = os.path.join(parent_dir, "pkl_data", model_folder)

raw_file = os.path.join(raw_path, f"{file_name}.tif")
mask_file = os.path.join(mask_path, f"{file_name}.tif")
pkl_file = os.path.join(pkl_path, f"{file_name}.pkl")

### **Load Data and Visualize**
Load raw image and mask, visualize in Napari, and filter for common cells.

In [ ]:
X = tifffile.imread(raw_file)
Y = tifffile.imread(mask_file)
print(f"Loaded raw image {X.shape} and mask {Y.shape}")

viewer.add_image(X, name="Raw")
viewer.add_labels(Y, name="Mask", opacity=0.35)

occurrence_limit = 80 # Include only cells that are present in at least 80% of frames
common_cells, _ = seg_utils.get_common_cells(Y, occurrence=occurrence_limit)
print(f"Common cells: {len(common_cells)}")

Y[~np.isin(Y, common_cells)] = 0
viewer.add_labels(Y, name="Filtered Mask", opacity=0.35)

### **Creating the points layer**

Before you advance with the next cell you need to press the button to create a *New points layer*.<br>
Continue with locating incorrectly placed labels, if they exist, by first clicking the "Add points"-button to then click the different incorrectly placed labels.<br>
Run the next cell to create a new `mask2` layer with the incorrectly placed labels removed.

In [ ]:
points_layer = viewer.layers[2].data.astype(np.uint16)
labels_at_points = Y[points_layer[:, 0], points_layer[:, 1], points_layer[:, 2]]
Y = np.where(np.isin(Y, labels_at_points), 0, Y)
viewer.add_labels(Y, name="Mask with Points Removed", opacity=0.35)

### **Plot Average Intensity**
Calculate and plot the average intensity over time.

In [ ]:
avg_intensity = np.mean(X, axis=(1, 2))
plt.figure(figsize=(15, 3))
plt.plot(avg_intensity, label="Average Intensity")
plt.legend()
plt.title("Average Intensity over Time")
plt.show()

### **Analyze Cells**
Compute intensities and center of mass data for all cells.

In [ ]:
def analyze_cells(X, Y, occurrence_limit=80, radius=10):
    d = {}
    com_mask_all = np.zeros(Y.shape, dtype=np.uint8)
    for c in tqdm(np.unique(Y)[1:], desc="Processing Cells"):
        com_mask, intensities_circle, com_coords = seg_utils.get_cell_intensities_circle(c, Y, X, radius)
        intensities = seg_utils.get_cell_intensities(c, Y, X)
        d[c] = {
            'intensities': intensities,
            'intensities_circle': intensities_circle,
            'com_coords': com_coords,
            'occurrence_limit': occurrence_limit,
            'circle_radius': radius
        }
        com_mask_all += com_mask
    viewer.add_labels(com_mask_all, name="All COM Masks")
    return d

d = analyze_cells(X, Y, occurrence_limit=occurrence_limit)

### **NaN Check and Save to Pickle**
Check for NaNs and save the results to a pickle file.

In [ ]:
for cell, data in d.items():
    for key in ['intensities', 'intensities_circle']:
        if np.any(np.isnan(data[key])):
            print(f"Warning: NaNs found in {key} for cell {cell}")

os.makedirs(pkl_path, exist_ok=True)
with open(pkl_file, 'wb') as f:
    pickle.dump(d, f)
print(f"Data saved to {pkl_file}")